# Handling missing data

## Operating on Null Values

As we have seen, Pandas treats None and NaN as essentially interchangeable for indicating missing or null values. To facilitate this convention, there are several useful methods for detecting, removing, and replacing null values in Pandas data structures. They are:

    isnull(): generate a boolean mask indicating missing values
    notnull(): opposite of isnull()
    dropna(): return a filtered version of the data
    fillna(): return a copy of the data with missing values filled or imputed

We will finish this section with a brief discussion and demonstration of these routines:

### Evaluating for Missing Data

At the base level, pandas offers two functions to test for missing data, **isnull()** and **notnull()**. As you may suspect, these are simple functions that return a boolean value indicating whether the passed in argument value is in fact missing data.

In addition to the above functions, pandas also provides two methods to check for missing data on Series and DataFrame objects. These methods evaluate each object in the Series or DataFrame and provide a boolean value indicating if the data is missing or not.

For example, let’s create a simple Series in pandas:

In [1]:
import pandas as pd
import numpy as np

s = pd.Series([2,3,np.nan,7,"The Hobbit"])

Now evaluating the Series s, the output shows each value as expected, including index 2 which we explicitly set as missing.

In [2]:
s

0             2
1             3
2           NaN
3             7
4    The Hobbit
dtype: object

To test the **isnull()** method on this series, we can use **s.isnull()** and view the output:

In [3]:
s.isnull()

0    False
1    False
2     True
3    False
4    False
dtype: bool

As expected, the only value evaluated as missing is index 2.
<hr>
### Determine if ANY Value in a Series is Missing

While the **isnull()** method is useful, sometimes we may wish to evaluate whether any value is missing in a Series.

There are a few possibilities involving chaining multiple methods together.

The fastest method is performed by chaining **.values.any()**:

In [4]:
s.isnull().values.any()

True

In some cases, you may wish to determine how many missing values exist in the collection, in which case you can use **.sum()** chained on:

In [5]:
s.isnull().sum()

1

### Count Missing Values in DataFrame

While the chain of **.isnull().values.any()** will work for a DataFrame object to indicate if any value is missing, in some cases it may be useful to also count the number of missing values across the entire DataFrame. Since DataFrames are inherently multidimensional, we must invoke two methods of summation.

For example, first we need to create a simple DataFrame with a few missing values:

In [6]:
df = pd.DataFrame(np.random.randn(5,5))

In [7]:
df

,0,1,2,3,4
0,-0.167214,0.627857,1.148771,-0.533198,-0.779749
1,1.526999,-0.469471,0.937033,0.534133,-1.383601
2,0.190699,0.352379,-2.009647,1.647401,0.235933
3,-1.231128,0.579533,-0.659043,0.096645,-0.589156
4,-0.024667,-1.610363,0.166223,0.009822,0.683680


In [8]:
 df[df > 0.9] = pd.np.nan

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead
  """Entry point for launching an IPython kernel.


In [9]:
df

,0,1,2,3,4
0,-0.167214,0.627857,NaN,-0.533198,-0.779749
1,NaN,-0.469471,NaN,0.534133,-1.383601
2,0.190699,0.352379,-2.009647,NaN,0.235933
3,-1.231128,0.579533,-0.659043,0.096645,-0.589156
4,-0.024667,-1.610363,0.166223,0.009822,0.683680


Now if we chain a **.sum()** method on, instead of getting the total sum of missing values, we’re given a list of all the summations of each column:

In [10]:
df.isnull().sum()

0    1
1    0
2    2
3    1
4    0
dtype: int64

We can see in this example, our first column contains three missing values, along with one each in column 2 and 3 as well.

In order to get the total summation of all missing values in the DataFrame, we chain two **.sum()** methods together:

In [11]:
df.isnull().sum().sum()

4

### Detecting Null Values

Pandas data structures have two useful methods for detecting null data: **isnull()** and **notnull()**. Either one will return a boolean mask over the data, for example:

In [12]:
data = pd.Series([1, np.nan, 'hello', None])

In [13]:
data.isnull()

0    False
1     True
2    False
3     True
dtype: bool

As mentioned in section X.X, boolean masks can be used directly as a Series or DataFrame index:

In [14]:
data[data.notnull()]

0        1
2    hello
dtype: object

The **isnull()** and **notnull()** methods produce similar boolean results for DataFrames.

### Dropping Null Values

In addition to the masking used above, there are the convenience methods, **dropna()** and **fillna()**, which respectively remove NA values and fill-in NA values. For a Series, the result is straightforward:

In [15]:
data.dropna()

0        1
2    hello
dtype: object

For a dataframe, there are more options. Consider the following dataframe:

In [16]:
df = pd.DataFrame([[1,      np.nan, 2],

                            [2,      3,      5],

                            [np.nan, 4,      6]])

df

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


We cannot drop single values from a DataFrame; we can only drop full rows or full columns. Depending on the application, you might want one or the other, so dropna() gives a number of options for a DataFrame.

By default, dropna() will drop all rows in which any null value is present:

In [17]:
df.dropna()

,0,1,2
1,2.0,3.0,5


Alternatively, you can drop NA values along a different axis: axis=1 drops all columns containing a null value:

In [18]:
df.dropna(axis=1)

,2
0,2
1,5
2,6


But this drops some good data as well; you might rather be interested in dropping rows or columns with all NA values, or a majority of NA values. This can be specified through the how or thresh parameters, which allow fine control of the number of nulls to allow through.

The default is how='any', such that any row or column (depending on the axis keyword) containing a null value will be dropped. You can also specify how='all', which will only drop rows/columns which are all null values:

In [19]:
df[3] = np.nan

df

df.dropna(axis=1, how='all')

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


Keep in mind that to be a bit more clear, you can use axis='rows' rather than axis=0 and axis='columns' rather than axis=1.

For finer-grained control, the thresh parameter lets you specify a minimum number of non-null values for the row/column to be kept:

In [20]:
df.dropna(thresh=3)

,0,1,2,3
1,2.0,3.0,5,NaN


Here the first and last row have been dropped, because they contain only two non-null values.
<hr>
### Filling Null Values

Sometimes rather than dropping NA values, you’d rather replace them with a valid value. This value might be a single number like zero, or it might be some sort of imputation or interpolation from the good values. You could do this in-place using the isnull() method as a mask, but because it is such a common operation Pandas provides the fillna() method, which returns a copy of the array with the null values replaced.

Consider the following Series:

In [21]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))

data

a    1.0
b    NaN
c    2.0
d    NaN
e    3.0
dtype: float64

We can fill NA entries with a single value, such as zero:

In [22]:
data.fillna("Bhavuk")

a         1
b    Bhavuk
c         2
d    Bhavuk
e         3
dtype: object

We can specify a forward-fill to propagate the previous value forward:

In [23]:
# forward-fill

data.fillna(method='ffill')

a    1.0
b    1.0
c    2.0
d    2.0
e    3.0
dtype: float64

Or we can specify a back-fill to propagate the next values backward:

In [24]:
# back-fill

data.fillna(method='bfill')

a    1.0
b    2.0
c    2.0
d    3.0
e    3.0
dtype: float64

For DataFrames, the options are similar, but we can also specify an axis along-which the fills take place:

In [25]:
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [26]:
df.fillna(method='ffill', axis=1)

,0,1,2,3
0,1.0,1.0,2.0,2.0
1,2.0,3.0,5.0,5.0
2,NaN,4.0,6.0,6.0


Notice that if a previous value is not available during a forward fill, the NA value remains.